# TGLC local: curvas de luz de todos los sectores

**Kernel: `Python (tglc)`** (entorno conda `tglc`, separado de `starry`).

El HLSP de TGLC en MAST solo tiene S5/S6 para estos objetos. Aquí se corre TGLC localmente
(`tglc.quick_lc.tglc_lc`) sobre los mismos sectores que tienen las curvas propias
(`Light_Curve_Files/`), para poder repetir la prueba de consistencia entre sectores con una
fotometría independiente.

Qué hace TGLC (Han & Brandt 2023):
- descarga un corte de las FFI con TESScut y el catálogo Gaia DR3 de la región;
- ajusta un ePSF por imagen y **resta el flujo de todas las estrellas Gaia vecinas** (descontaminación);
- da dos fotometrías del objetivo: `psf_flux` (ajuste de PSF) y `aperture_flux` (apertura 3×3 sobre la
  imagen descontaminada), y sus versiones `cal_*` con la mediana fijada al flujo predicho por Gaia y
  sin tendencias lentas.

Salida de este notebook:
- `tglc_local/TIC_<tic>/lc/*.fits`: archivos de TGLC tal cual;
- `Light_Curve_Files_TGLC/local/tic <tic>_S_<s>_TGLC.txt`: tabla de texto con `time` y las cuatro
  fotometrías, que lee el notebook `1d_tglc_profundidades.ipynb` (kernel `starry`);
- `tglc_local/registro_tglc.csv`: estado de cada (TIC, sector).

**Sectores de 200 s (S87, S98):** el ajuste del ePSF es imagen por imagen y domina el tiempo. Con
`BIN_200S = 3` se promedian de a 3 imágenes buenas *antes* del ajuste (`tglc_local/tglc_binning.py`),
lo que deja la cadencia en 600 s (como S32) y divide el tiempo por 3. Validado en TIC 138829403 S6:
binear antes del ePSF y binear la curva después dan la misma profundidad (D1 = 0.0488 en las cuatro
fotometrías). Se mantiene el mismo corte de 90 px en todos los sectores para no introducir una
diferencia de modelo de fondo entre sectores.

**Disco:** el corte de S6 (30 min) ocupa ~0.7 GB y el de S98 (200 s) unas 10 veces más. Con
`BORRAR_CORTES = True` se borra cada corte en cuanto se extrae su curva.

In [ ]:
import os
import glob
import time
import traceback
import warnings
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
from astropy.io import fits

RUTA_ORION = Path("/home/luciam/Documents/starry/Resultados_Orion")
DIR_LOCAL = RUTA_ORION / "tglc_local"                       # trabajo de TGLC (cortes, ePSF, lc)
DIR_TXT = RUTA_ORION / "Light_Curve_Files_TGLC" / "local"    # curvas exportadas para el kernel starry
DIR_LOCAL.mkdir(exist_ok=True); DIR_TXT.mkdir(parents=True, exist_ok=True)
REGISTRO = DIR_LOCAL / "registro_tglc.csv"

SOLO_REVISAR = True     # True: solo los TIC con alguna marca distinta de "consistente"; False: todos
SIZE = 90               # lado del corte en píxeles (máx. 100); más grande = mejor fondo/ePSF
LIMIT_MAG = 16          # estrellas Gaia más brillantes que esto entran al modelo de la escena
N_PROC = 6              # corridas en paralelo (cada una usa 1 hilo de BLAS y unos GB de RAM)
BIN_200S = 3            # sectores de 200 s (S56+): promediar de a 3 imágenes antes del ePSF (600 s)
BORRAR_CORTES = True    # borrar source/*.pkl al terminar cada sector
REHACER = False         # True: volver a correr aunque ya exista la curva exportada

## 1) Lista de (TIC, sector)

Los sectores son los mismos de `priors_v2.csv`, es decir, los de las curvas propias.

In [ ]:
v2 = pd.read_csv(RUTA_ORION / "priors_v2.csv")
cons = pd.read_csv(RUTA_ORION / "resultados_pipeline" / "profundidades_v2" / "consistencia_sectores.csv")
if SOLO_REVISAR:
    tics = cons.loc[(cons["marca_primario"] != "consistente")
                    | (cons["marca_secundario"] == "difiere_revisar"), "TIC"].astype(int).tolist()
else:
    tics = sorted(v2["TIC"].astype(int).unique())
tareas = [(int(r.TIC), int(r.Sector)) for r in v2.itertuples() if int(r.TIC) in tics]


cat = pd.read_csv(RUTA_ORION / "resultados_pipeline" / "resumen_por_tic_combinado.csv")
COORD = {int(r.TIC): (float(r.RA), float(r.DEC)) for r in cat.itertuples()}   # respaldo para cruces ambiguos


def ruta_txt(tic, sector):
    return DIR_TXT / f"tic {tic}_S_{sector}_TGLC.txt"


# primero los sectores más largos (200 s, luego 10 min), para repartir mejor la carga entre procesos
costo = lambda t: 3 if t[1] >= 56 else (2 if t[1] >= 27 else 1)
pendientes = sorted([t for t in tareas if REHACER or not ruta_txt(*t).exists()], key=costo, reverse=True)
print(f"{len(tics)} TIC, {len(tareas)} curvas; pendientes: {len(pendientes)}")
pd.DataFrame(tareas, columns=["TIC", "Sector"]).groupby("TIC")["Sector"].apply(list)

## 2) Correr TGLC y exportar

`correr(tic, sector)` descarga el corte (`ffi_cut`, a lo más `N_DESCARGAS` a la vez y con reintentos,
porque TESScut responde 502/503 si se le piden muchos cortes juntos), ajusta el ePSF (`epsf`), busca el FITS del objetivo en
`tglc_local/TIC_<tic>/lc/`, lo exporta a texto y borra el corte. Cada corrida escribe su propio log en
`tglc_local/TIC_<tic>/log_S<s>.txt`. Si una falla (p. ej. MAST o Gaia caídos), se registra el error y
basta con volver a correr la celda: solo se repiten las pendientes.

Las curvas exportadas conservan todos los puntos; las banderas (`TESS_flags`, `TGLC_flags`) se aplican
en el notebook de medición.

In [ ]:
COLUMNAS = ["time", "psf_flux", "aperture_flux", "cal_psf_flux", "cal_aper_flux", "background",
            "TESS_flags", "TGLC_flags"]


def exportar(fits_path, tic, sector):
    with fits.open(fits_path) as h:
        h0, d = h[0].header, h[1].data
        tab = pd.DataFrame({c: np.asarray(d[c]).astype(float) for c in COLUMNAS})
        encabezado = (f"TIC {tic}  sector {sector}  Gaia DR3 {h0['GAIADR3']}  TESSMAG {h0['TESSMAG']:.3f}  "
                      f"archivo {Path(fits_path).name}")
    with open(ruta_txt(tic, sector), "w") as f:
        f.write(f"# {encabezado}\n")
        tab.to_csv(f, sep=" ", index=False, float_format="%.10g")
    return dict(gaia=int(h0["GAIADR3"]), tmag=float(h0["TESSMAG"]), n=len(tab))


import multiprocessing as mp

N_DESCARGAS = 2         # pedidos simultáneos a TESScut/Gaia (con más, MAST responde 502/503/504)
N_INTENTOS = 16         # reintentos de descarga: espera 1, 2, 4, 8, 16 y luego 30 min (tolera ~7 h de caída)
DESCARGA = mp.Semaphore(N_DESCARGAS)   # se hereda en los procesos hijos (fork)


def correr(tic, sector):
    import sys
    from contextlib import redirect_stdout, redirect_stderr
    sys.path.insert(0, str(DIR_LOCAL))
    from tglc_binning import descargar_source, ajustar
    carpeta = DIR_LOCAL / f"TIC_{tic}"
    carpeta.mkdir(exist_ok=True)
    t_ini = time.time()
    n_bin = BIN_200S if sector >= 56 else 1
    fila = dict(TIC=tic, Sector=sector, bin=n_bin)
    try:
        with open(carpeta / f"log_S{sector}.txt", "w") as log, redirect_stdout(log), redirect_stderr(log), \
                warnings.catch_warnings():
            warnings.simplefilter("ignore")
            for intento in range(N_INTENTOS):
                try:
                    with DESCARGA:
                        source = descargar_source(tic, str(carpeta) + "/", sector, size=SIZE, limit_mag=LIMIT_MAG)
                    break
                except Exception as e:
                    if intento == N_INTENTOS - 1:
                        raise
                    espera = min(2 ** intento, 30)
                    print(f"descarga falló ({type(e).__name__}: {e}); reintento en {espera} min", flush=True)
                    time.sleep(60 * espera)
            fila["minutos_descarga"] = (time.time() - t_ini) / 60
            salidas = ajustar(source, tic, str(carpeta) + "/", n=n_bin, limit_mag=LIMIT_MAG,
                              ra=COORD[tic][0], dec=COORD[tic][1])
        # epsf solo escribe la curva del objetivo; si no devolvió rutas, se busca por TICID en lc/
        mio = [p for p in salidas if f"-s{sector:04d}-" in str(p)]
        if not mio:
            cand = glob.glob(str(carpeta / "lc" / "**" / f"*-s{sector:04d}-*.fits"), recursive=True)
            mio = [p for p in cand if str(fits.getheader(p).get("TICID")) == str(tic)]
        if not mio:
            raise RuntimeError(f"TGLC no produjo la curva del objetivo ({len(cand)} archivos del sector)")
        fila.update(estado="ok", archivo=Path(mio[0]).name, **exportar(mio[0], tic, sector))
    except Exception as e:
        fila.update(estado="error", error=f"{type(e).__name__}: {e}",
                    detalle=traceback.format_exc(limit=3).replace("\n", " | "))
    finally:
        if BORRAR_CORTES and fila.get("estado") == "ok":   # si falló, se conserva para reintentar sin bajar
            for p in glob.glob(str(carpeta / "source" / f"*sector_{sector}.pkl")):
                os.remove(p)
    fila["minutos"] = (time.time() - t_ini) / 60
    return fila


filas = []
with ProcessPoolExecutor(max_workers=N_PROC) as ex:
    futuros = {ex.submit(correr, *t): t for t in pendientes}
    for fut in as_completed(futuros):
        fila = fut.result()
        filas.append(fila)
        print(f"TIC {fila['TIC']:>10} S{fila['Sector']:<3} {fila['estado']:5} {fila['minutos']:5.1f} min"
              + (f"  {fila.get('error', '')}" if fila["estado"] != "ok" else f"  Tmag {fila['tmag']:.2f}"))

reg = pd.concat([pd.read_csv(REGISTRO), pd.DataFrame(filas)]) if REGISTRO.exists() else pd.DataFrame(filas)
reg = reg.drop_duplicates(["TIC", "Sector"], keep="last").sort_values(["TIC", "Sector"])
reg.to_csv(REGISTRO, index=False)
reg

## 3) Control rápido

Curva cruda de cada (TIC, sector) exportado. Sirve para detectar sectores donde TGLC falló
(saturación, borde del detector, fondo de la nebulosa) antes de medir profundidades.
Ojo: **TIC 50897998 tiene Tmag ≈ 5.5 y está saturada**; TGLC no es confiable ahí.

In [ ]:
import matplotlib.pyplot as plt

archivos = sorted(DIR_TXT.glob("tic *_S_*_TGLC.txt"))
fig, axs = plt.subplots(len(archivos), 1, figsize=(12, 1.6 * len(archivos)), squeeze=False)
for ax, p in zip(axs[:, 0], archivos):
    d = pd.read_csv(p, sep=" ", comment="#")
    ok = (d["TGLC_flags"] == 0) & (d["TESS_flags"] == 0)
    for c, col in (("cal_psf_flux", "k"), ("cal_aper_flux", "#8e44ad")):
        y = d.loc[ok, c] / np.nanmedian(d.loc[ok, c])
        ax.plot(d.loc[ok, "time"], y, ".", ms=1, color=col, alpha=0.5)
    ax.text(0.01, 0.05, p.stem.replace("tic ", "TIC "), transform=ax.transAxes, fontsize=8)
fig.tight_layout()
plt.show()